# MNIST Digit Image Classification

In [21]:
# !pip install idx2numpy
import numpy as np
import matplotlib.pyplot as plt
import idx2numpy
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Prepare and Preprocess the Datasets
Use idx2numpy library to abstract away the file handling, then transform the numpy arrays into something PyTorch can use.

In [22]:
# Import training and testing datasets and labels
training_images = idx2numpy.convert_from_file('train-images.idx3-ubyte')
training_labels = idx2numpy.convert_from_file('train-labels.idx1-ubyte')
testing_images = idx2numpy.convert_from_file('t10k-images.idx3-ubyte')
testing_labels = idx2numpy.convert_from_file('t10k-labels.idx1-ubyte')

# Check shapes
print(f"Train images shape: {training_images.shape}, Train labels shape: {training_labels.shape}")
print(f"Test images shape: {testing_images.shape}, Test labels shape: {testing_labels.shape}")

# Normalize training and test images from [0, 255] to [-0.1, 1.175] to accomodate LeNet-5 Implementation (This is mentioned page 7 in their article)
training_images = (training_images / 255.0) * 1.185 - 0.1
testing_images = (testing_images / 255.0) * 1.185 - 0.1


# Originally planned to stratify, but the proportion of digits in training and testing sets are close enough that I don't think it is necessary
#
# Split the training images into 10 numpy array based on corresponding labels (This is for stratification as I believe randomly pulling from the )
# train_images_split = []
# test_images_split = []
# for i in range(10):
#   train_images_split.append(train_images[train_labels == i])
#   test_images_split.append(test_images[test_labels == i])

# Check shapes (There seems to be around ~6000 images for each number except 1 which has around 7000)
# for i in range(10):
#   print(f"Train images shape for label {i}: {100 * train_images_split[i].shape[0] / 60000:.2f}")
#   print(f"Test images shape for label {i}: {100 * test_images_split[i].shape[0] / 10000:.2f}")



# Convert to torch tensors and add channel dimension (This is so it work with pytorch functions and is because grayscale = 1 channel)
training_images = torch.tensor(training_images, dtype=torch.float32).unsqueeze(1)
training_labels = torch.tensor(training_labels, dtype=torch.long)
testing_images = torch.tensor(testing_images, dtype=torch.float32).unsqueeze(1)
testing_labels = torch.tensor(testing_labels, dtype=torch.long)

# Preprocess a 1 dimensional version of the images to prevent future processing during FFNN
training_images_flat = training_images.view(training_images.size(0), -1)
testing_images_flat = testing_images.view(testing_images.size(0), -1)
# print(f"Train images shape: {train_images_flat.shape}, Train labels shape: {train_labels.shape}")
# print(f"Test images shape: {test_images_flat.shape}, Test labels shape: {test_labels.shape}")

# Create PyTorch datasets to put into a data loader
training_dataset_flat = TensorDataset(training_images_flat, training_labels) # For FFNN
training_dataset = TensorDataset(training_images, training_labels) # For CNN

# Create data loaders to make handling data simple and easy
training_loader_flat = DataLoader(dataset=training_dataset_flat, batch_size=256, shuffle=True) # For FFNN
training_loader = DataLoader(dataset=training_dataset, batch_size=256, shuffle=True) # For CNN

# Display an Image to check data was loaded properly
# images, labels = next(iter(train_loader))
# print(f"Image batch shape: {images.shape}, Label batch shape: {labels.shape}")

# plt.imshow(images[0].squeeze(), cmap='gray')
# plt.title(f"Label: {labels[0].item()}")
# plt.show()

Train images shape: (60000, 28, 28), Train labels shape: (60000,)
Test images shape: (10000, 28, 28), Test labels shape: (10000,)


In [23]:
# GPU acceleration for pytorch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Train a feedforward neural network
Needs to have at least 2 hidden layers.

In [24]:
# Define the Feed Forward Neural Network Class
class FeedForwardNN(nn.Module):
  def __init__(self):
      super(FeedForwardNN, self).__init__()

      # 5 Fully Connected Layers, dimensions are halved each successive layer
      # 784 comes from 28 * 28
      self.fc1 = nn.Linear(784, 392)  # Decreasing layer size to encourage dimensionality reduction, idk halving just makes me happy
      self.fc2 = nn.Linear(392, 196)
      self.fc3 = nn.Linear(196, 98)
      self.fc4 = nn.Linear(98, 49)
      self.fc5 = nn.Linear(49, 10)

  def forward(self, x):
      x = F.relu(self.fc1(x)) # Simple ReLu activation function
      x = F.relu(self.fc2(x))
      x = F.relu(self.fc3(x))
      x = F.relu(self.fc4(x))
      x = self.fc5(x)
      return x

In [25]:
# Bundle everything into a function for the final testing runs
def FFNN(device, training_loader_flat):
  # Initialize the model
  model = FeedForwardNN()

  # Define the loss function and optimizer
  criterion = nn.CrossEntropyLoss() # Loss function for Multiclass classification
  optimizer = optim.SGD(model.parameters(), lr=0.1) # Use vanilla stochastic gradient descent because that is what we learned in class

  # GPU acceleration
  model.to(device)

  # Default epoch x dataloaders training loop
  num_epochs = 5 # 5 epochs and batch size 256 seems to toe the 95% threshold, while taking the least time (More epochs and smaller batch size results in higher testing scores)
  for epoch in range(num_epochs):
      model.train()
      # cumulative_loss = 0.0

      # Loop through the flat 1 dimensional dataloader
      for images, labels in training_loader_flat:

          # Load the images and labels to the GPU
          images, labels = images.to(device), labels.to(device)

          # Forward pass
          outputs = model(images)

          # Calculate Loss
          loss = criterion(outputs, labels)

          # Backward pass and optimization
          loss.backward() # Build gradients
          optimizer.step() # Update weights and biases
          optimizer.zero_grad() # Zero out the previous gradients

          # cumulative_loss += loss.item()

      # epoch_loss = cumulative_loss / len(train_loader)
      # print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

  return model

# testmodel = FFNN(device, training_loader_flat)

In [26]:
# Bundle everything into a function for the final testing runs
def test_FFNN(model, device, testing_images_flat, testing_labels):

  correct = 0
  with torch.no_grad(): # Turn off gradient calculations
      model.eval()

      # Load the images and labels to the GPU
      images, labels = testing_images_flat.to(device), testing_labels.to(device)

      # Make the Classifications
      outputs = model(images)

      # Choose 0-9 based on which of the 10 outputs has the highest value for each data point
      _, predicted = torch.max(outputs.data, 1)

      # Compare the Classification outputs against the labels and add up the total
      correct = (predicted == labels).sum().item()

  accuracy = 100 * correct / 10000
  # print(f"\nTest Accuracy: {accuracy:.2f}%")
  return accuracy

# test_FFNN(testmodel, device, testing_images_flat, testing_labels)

# Create a convolution neural network
Needs to have at least 2 convolution layers and 2 fully connected layers. In this case, I am implemented the LeNet-5 Architecture dicussed in the "**Gradient-based learning applied to document recognition**" paper instead of creating my own.

In [27]:
# Define the Convolution Neural Network Class
class LeNet5(nn.Module):
  def __init__(self):
    super(LeNet5, self).__init__()
    # This is the LeNet-5 Architecture mentioned in the article
    self.conv1 = nn.Conv2d(1, 6, kernel_size=5, stride=1, padding=2) # C1, The first Convolution Layer needs padding=2 because LeNet-5 starts with a 32x32 image and we start with a 28 by 28
    self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1) # C3
    self.conv3 = nn.Conv2d(16, 120, kernel_size=5, stride=1) # C5

    # Custom Pooling layer for LeNet-5 (The paper mentions the need for trainable weights and biases)
    self.avgpool1 = TrainablePoolingCoeff(num_channels=6) # S2
    self.avgpool2 = TrainablePoolingCoeff(num_channels=16) # S4

    # Fully Connected Layers
    # Post processing after Convolution layers for classification
    self.fc1 = nn.Linear(120, 84) # F6
    self.fc2 = nn.Linear(84, 10) # F7

  def scaled_tanh(self, x):
    # This is something the authors of the paper included to supposedly speed up convergence
    return 1.1759 * torch.tanh((2/3) * x)

  def forward(self, x):
    # Idea is:
    # Pixels to Spatial Dimensions
    # Spatial Dimensions to Features
    # Features to Classification

    # First Convolution and Pooling
    x = self.scaled_tanh(self.conv1(x))
    x = self.avgpool1(x)
    # print(f"Shape after conv1 + pool: {x.shape}")

    # Second Convolution and Pooling
    x = self.scaled_tanh(self.conv2(x))
    x = self.avgpool2(x)
    # print(f"Shape after conv2 + pool: {x.shape}")

    # Third Convolution
    x = self.scaled_tanh(self.conv3(x))
    # print(f"Shape after conv3: {x.shape}")

    # Flatten to features
    x = x.view(x.size(0), -1)
    # print(f"Shape after flattening: {x.shape}")

    # Classification from Features
    x = self.scaled_tanh(self.fc1(x))
    x = self.fc2(x)
    return x

# In page 7 of the paper they say that do average pooling and multiplies it with a trainable coefficient and adds a trainable bias
# Apparently this has multiple uses such as being not as sensitive to translations or distortions, etc.
class TrainablePoolingCoeff(nn.Module):
  def __init__(self, num_channels):
    super(TrainablePoolingCoeff, self).__init__()
    self.trainable_coeff = nn.Parameter(torch.ones(num_channels))
    self.trainable_bias = nn.Parameter(torch.zeros(num_channels))

  def forward(self, x):
    x = F.avg_pool2d(x, kernel_size=2, stride=2)
    # print(f"Shape after avg pool: {x.shape}")
    x = x * self.trainable_coeff.view(1, -1, 1, 1) + self.trainable_bias.view(1, -1, 1, 1)
    return x

In [28]:
# Bundle everything into a function for the final testing runs
def CNN(device, training_loader):
  # Initialize the model
  model = LeNet5()

  # Define the loss function and optimizer
  criterion = nn.CrossEntropyLoss() # MLE was mentioned in the paper and was compared to MSE, but I couldn't get MSE to work and ran out of time so I changed it back to Cross Entropy Loss
  optimizer = optim.SGD(model.parameters(), lr=0.1) # The vanilla Stochastic Gradient Descent was specified

  # GPU acceleration
  model.to(device)

  num_epochs = 5
  for epoch in range(num_epochs):
      model.train()
      # cumulative_loss = 0.0

      for images, labels in training_loader:

          # Load the images and labels to the GPU
          images, labels = images.to(device), labels.to(device)

          # Forward pass
          outputs = model(images)

          # Calculate Loss
          loss = criterion(outputs, labels)

          # Backward pass and optimization
          loss.backward() # Build gradients
          optimizer.step() # Update weights and biases
          optimizer.zero_grad() # Zero out the previous gradients

      #     cumulative_loss += loss.item()

      # epoch_loss = cumulative_loss / len(train_loader)
      # print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

  return model

# testmodel2 = CNN(device, training_loader)

In [29]:
# Bundle everything into a function for the final testing runs
def test_CNN(model, device, testing_images, testing_labels):

  correct = 0
  with torch.no_grad(): # Turn off gradient calculations
      model.eval()

      # Load the images and labels to the GPU
      images, labels = testing_images.to(device), testing_labels.to(device)
      outputs = model(images)

      # Choose 0-9 based on which of the 10 outputs has the highest value for each data point
      _, predicted = torch.max(outputs.data, 1)

      # Compare the Classification outputs against the labels and add up the total
      correct = (predicted == labels).sum().item()

  accuracy = 100 * correct / 10000
  # print(f"\nTest Accuracy: {accuracy:.2f}%")
  return accuracy

# test_CNN(testmodel2, device, testing_images, testing_labels)

# Testing runs
Trains and tests the feedforward neural network and convolutional neural network implementation 5 times and outputs the average accuracy.

In [30]:
# Final Testing Runs
fnn_results = []
cnn_results = []
for i in range(5):
  print(f"Testing Run {i+1}\n")

  print("Training Feedforward Neural Network...")
  model1 = FFNN(device, training_loader_flat)
  accuracy1 = test_FFNN(model1, device, testing_images_flat, testing_labels)
  fnn_results.append(accuracy1)
  print(f"FFNN Test Accuracy: {accuracy1:.2f}%\n")

  print("Training Convolution Neural Network...")
  model2 = CNN(device, training_loader)
  accuracy2 = test_CNN(model2, device, testing_images, testing_labels)
  cnn_results.append(accuracy2)
  print(f"CNN Test Accuracy: {accuracy2:.2f}%\n")

print("Average Results:")
print(f"FFNN Average Accuracy: {np.mean(fnn_results):.2f}%")
print(f"CNN Average Accuracy: {np.mean(cnn_results):.2f}%")

Testing Run 1

Training Feedforward Neural Network...
FFNN Test Accuracy: 95.49%

Training Convolution Neural Network...
CNN Test Accuracy: 96.59%

Testing Run 2

Training Feedforward Neural Network...
FFNN Test Accuracy: 96.17%

Training Convolution Neural Network...
CNN Test Accuracy: 96.36%

Testing Run 3

Training Feedforward Neural Network...
FFNN Test Accuracy: 95.17%

Training Convolution Neural Network...
CNN Test Accuracy: 96.52%

Testing Run 4

Training Feedforward Neural Network...
FFNN Test Accuracy: 95.32%

Training Convolution Neural Network...
CNN Test Accuracy: 96.21%

Testing Run 5

Training Feedforward Neural Network...
FFNN Test Accuracy: 95.61%

Training Convolution Neural Network...
CNN Test Accuracy: 96.25%

Average Results:
FFNN Average Accuracy: 95.55%
CNN Average Accuracy: 96.39%


# Report
First, my thoughts on the feedforward neural network is that it is quite forgiving. I played with several different configurations such as 2-7 hidden layers, less/more nodes in the hidden layers, lower number of epochs, larger/smaller batch sizes, and most of them were able to reach 95% accuracy or greater with around 5 to 10 epochs. In the end, I landed on a batch size of 256 and 5 epochs to push training time down to around 15-20 seconds and I left whatever made me happy in the hidden layers.

Next, the LeNet-5 implementation for the convolutional neural network was more difficult for me. I looked up the article at *https://ieeexplore.ieee.org/document/726791* read through the implementation and normalized [0, 255] to [-0.1, 1.175], implemented a average pooling function/class with trainable weights and biases, and a scaled tanh activation to improve convergence speed. Here the algorithm was taking 4-5 minutes at 10 epochs, batch size 64, and learning rate 0.01. At this point it was a lot of toying with batch size, epochs, and learning rate to try and get it to converge faster. My goal was to get to 95% accuracy as fast as possible and I landed on the parameters: batch size = 256, learning rate = 0.1 and epoch = 5 (training time is now around 1:30 to 2 minutes). I believe the reason these changes worked is because a larger batch size both allows for more hardware acceleration and produces less noisy gradients, less noisy gradients allows for larger step sizes (learning rate), and the epoch I started at 10 and just kept lowering until I could not reliably get 95% accuracy and landed at 5 epochs.